<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3 插曲：Chisel 标准库
**上一步：[生成器：集合](3.2_collections.ipynb)**<br>
**下一步：[高阶函数](3.3_higher-order_functions.ipynb)**

## 动机
Chisel 的核心在于重用，因此提供一个标准接口库（鼓励 RTL 的互操作性）和常用硬件模块的生成器是理所当然的。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 速查表
[Chisel3 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf) 包含了所有主要硬件构建 API 的摘要，包括我们下面将介绍的一些标准库实用程序。

# Decoupled：一个标准的 Ready-Valid 接口
Chisel 提供的常用接口之一是 `DecoupledIO`，它为数据传输提供了一个 ready-valid 接口。其思想是，源端用要传输的数据驱动 `bits` 信号，并在有数据要传输时驱动 `valid` 信号。宿端在准备好接收数据时驱动 `ready` 信号，当 `ready` 和 `valid` 在一个周期内同时有效时，数据被认为已传输。

这为数据传输提供了双向流控制机制，包括反压机制。

注意：`ready` 和 `valid` 不应组合耦合，否则可能导致不可综合的组合环路。`ready` 仅应取决于宿端是否能够接收数据，而 `valid` 仅应取决于源端是否有数据。只有在事务完成（在下一个时钟周期）后，值才应更新。

任何 Chisel 数据都可以封装在 `DecoupledIO` 中（用作 `bits` 字段），如下所示：

```scala
val myChiselData = UInt(8.W)
// 或任何 Chisel 数据类型，例如 Bool()、SInt(...)，甚至自定义 Bundle
val myDecoupled = Decoupled(myChiselData)
```

上面创建了一个新的 `DecoupledIO` Bundle，包含以下字段：
- `valid`: Output(Bool)
- `ready`: Input(Bool)
- `bits`: Output(UInt(8.W))
___

本节的其余部分结构与之前略有不同：我们将给出一些代码示例和打印电路状态的测试用例，而不是给您编码练习。在运行测试之前，尝试预测将打印什么内容。

## 队列

`Queue` 创建一个 FIFO（先进先出）队列，两侧都有 Decoupled 接口，允许反压。数据类型和元素数量都是可配置的。

>请注意，前几次我们打印 `peek` 出来的值时，我们对其调用了 `litValue`。这将返回的 chisel 字面量转换为 `BigInt`。之后类似的调用我们没有调用 `litValue`，您可以看到有关 `peek` 返回值的更多信息，例如类型和宽度。

In [ ]:
test(new Module {
    // 使用队列的示例电路
    val io = IO(new Bundle {
      val in = Flipped(Decoupled(UInt(8.W)))
      val out = Decoupled(UInt(8.W))
    })
    val queue = Queue(io.in, 2)  // 2 元素队列
    io.out <> queue
  }) { c =>
    c.io.out.ready.poke(false.B)
    c.io.in.valid.poke(true.B)  // 入队一个元素
    c.io.in.bits.poke(42.U)
    println(s"开始：")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(true.B)  // 入队另一个元素
    c.io.in.bits.poke(43.U)
    // 您认为 io.out.valid 和 io.out.bits 会是什么？
    println(s"第一次入队后：")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(true.B)  // 读取一个元素，尝试入队
    c.io.in.bits.poke(44.U)
    c.io.out.ready.poke(true.B)
    // 您认为 io.in.ready 会是什么，这次入队会成功吗，会读取什么？
    println(s"第一次读取时：")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(false.B)  // 读出元素
    c.io.out.ready.poke(true.B)
    // 您认为这里会读取什么？
    println(s"第二次读取时：")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    // 第三次读取会产生任何东西吗？
    println(s"第三次读取时：")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)
}

## 仲裁器
仲裁器根据优先级将数据从 _n_ 个 `DecoupledIO` 源路由到一个 `DecoupledIO` 宿。
Chisel 中包含两种类型：
- `Arbiter`：优先处理索引较低的生产者
- `RRArbiter`：按轮询顺序运行

请注意，仲裁器路由是在组合逻辑中实现的。

以下示例将演示优先级仲裁器的使用（您也将在下一节中实现它）：

In [ ]:
test(new Module {
    // 使用优先级仲裁器的示例电路
    val io = IO(new Bundle {
      val in = Flipped(Vec(2, Decoupled(UInt(8.W))))
      val out = Decoupled(UInt(8.W))
    })
    // 仲裁器没有方便的构造函数，因此它的构建方式与任何模块相同
    val arbiter = Module(new Arbiter(UInt(8.W), 2))  // 2 对 1 优先级仲裁器
    arbiter.io.in <> io.in
    io.out <> arbiter.io.out
  }) { c =>
    c.io.in(0).valid.poke(false.B)
    c.io.in(1).valid.poke(false.B)
    c.io.out.ready.poke(false.B)
    println(s"开始：")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(1).valid.poke(true.B)  // 有效输入 1
    c.io.in(1).bits.poke(42.U)
    c.io.out.ready.poke(true.B)
    // 您认为输出会是什么？
    println(s"有效输入 1：")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(0).valid.poke(true.B)  // 有效输入 0 和 1
    c.io.in(0).bits.poke(43.U)
    // 您认为输出会是什么？哪些输入会准备就绪？
    println(s"有效输入 0 和 1：")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(1).valid.poke(false.B)  // 有效输入 0
    // 您认为输出会是什么？
    println(s"有效输入 0：")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
}

# 其他功能块
Chisel Utils 有一些执行无状态函数的辅助程序。

## 位操作实用程序
### PopCount
PopCount 以 `UInt` 形式返回输入中高位 (1) 的数量。

### Reverse
Reverse 返回位反转的输入。

In [ ]:
test(new Module {
    // 使用 PopCount 的示例电路
    val io = IO(new Bundle {
      val in = Input(UInt(8.W))
      val out = Output(UInt(8.W))
    })
    io.out := PopCount(io.in)
  }) { c =>
    // Integer.parseInt 用于从二进制规范创建整数
    c.io.in.poke(Integer.parseInt("00000000", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("00001111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("11001010", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("11111111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

}

In [ ]:
test(new Module {
    // 使用 Reverse 的示例电路
    val io = IO(new Bundle {
      val in = Input(UInt(8.W))
      val out = Output(UInt(8.W))
    })
    io.out := Reverse(io.in)
  }) { c =>
    // Integer.parseInt 用于从二进制规范创建整数
    c.io.in.poke(Integer.parseInt("01010101", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("00001111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("11110000", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("11001010", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")
}

## OneHot 编码实用程序
OneHot 是一种整数编码，其中每个值都有一条线，并且恰好有一条线为高电平。这允许高效地创建某些函数，例如多路选择器。但是，如果未保持单线高电平条件，则行为可能未定义。

以下两个函数提供二进制 (`UInt`) 和 OneHot 编码之间的转换，并且它们互为逆函数：
- UInt 转 OneHot：`UIntToOH`
- OneHot 转 UInt：`OHToUInt`

In [ ]:
test(new Module {
    // 使用 UIntToOH 的示例电路
    val io = IO(new Bundle {
      val in = Input(UInt(4.W))
      val out = Output(UInt(16.W))
    })
    io.out := UIntToOH(io.in)
  }) { c =>
    c.io.in.poke(0.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(1.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(8.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(15.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")
}

In [ ]:
test(new Module {
    // 使用 OHToUInt 的示例电路
    val io = IO(new Bundle {
      val in = Input(UInt(16.W))
      val out = Output(UInt(4.W))
    })
    io.out := OHToUInt(io.in)
}) { c =>
    c.io.in.poke(Integer.parseInt("0000 0000 0000 0001".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("0000 0000 1000 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("1000 0000 0000 0001".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    // 一些无效输入：
    // 无高电平
    c.io.in.poke(Integer.parseInt("0000 0000 0000 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    // 多个高电平
    c.io.in.poke(Integer.parseInt("0001 0100 0010 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")
}


## 多路选择器
这些多路选择器接受一个包含选择信号的值列表，并输出与最低索引选择信号关联的值。

这些可以接受 (select: Bool, value: Data) 元组列表，或相应的 select 和 value 列表作为参数。为简单起见，以下示例仅演示第二种形式。

### 优先级多路选择器
`PriorityMux` 输出与最低索引断言选择信号关联的值。

### OneHot 多路选择器
`Mux1H` 在保证恰好一个选择信号为高电平的情况下提供高效实现。如果假设不成立，则行为未定义。

In [ ]:
test(new Module {
    // 使用 PriorityMux 的示例电路
    val io = IO(new Bundle {
      val in_sels = Input(Vec(2, Bool()))
      val in_bits = Input(Vec(2, UInt(8.W)))
      val out = Output(UInt(8.W))
    })
    io.out := PriorityMux(io.in_sels, io.in_bits)
  }) { c =>
    c.io.in_bits(0).poke(10.U)
    c.io.in_bits(1).poke(20.U)

    // 仅选择较高索引
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // 同时选择 - 需要仲裁
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // 仅选择较低索引
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")
}

In [ ]:
test(new Module {
    // 使用 Mux1H 的示例电路
    val io = IO(new Bundle {
      val in_sels = Input(Vec(2, Bool()))
      val in_bits = Input(Vec(2, UInt(8.W)))
      val out = Output(UInt(8.W))
    })
    io.out := Mux1H(io.in_sels, io.in_bits)
  }) { c =>
    c.io.in_bits(0).poke(10.U)
    c.io.in_bits(1).poke(20.U)

    // 选择索引 1
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // 选择索引 0
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // 不选择（无效）
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // 同时选择（无效）
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")
}

## 计数器
`Counter` 是一个计数器，每个周期可以递增一次，直到达到某个指定的限制，此时它会溢出。请注意，它**不是**一个模块，并且其值是可访问的。

In [ ]:
test(new Module {
    // 使用 Mux1H 的示例电路
    val io = IO(new Bundle {
      val count = Input(Bool())
      val out = Output(UInt(2.W))
    })
    val counter = Counter(3)  // 3 计数计数器（输出范围 [0...2]）
    when(io.count) {
      counter.inc()
    }
    io.out := counter.value
  }) { c =>
    c.io.count.poke(true.B)
    println(s"开始：计数器值=${c.io.out.peek().litValue}")

    c.clock.step(1)
    println(s"步骤 1：计数器值=${c.io.out.peek().litValue}")

    c.clock.step(1)
    println(s"步骤 2：计数器值=${c.io.out.peek().litValue}")

    c.io.count.poke(false.B)
    c.clock.step(1)
    println(s"不递增的步骤：计数器值=${c.io.out.peek().litValue}")

    c.io.count.poke(true.B)
    c.clock.step(1)
    println(s"再次执行步骤：计数器值=${c.io.out.peek().litValue}")
}

---
# 您已完成！

[返回顶部。](#top)